In [329]:
import numpy as np
import pandas as pd
import matplotlib as plt

In [330]:
df = pd.read_csv('./data/GA_2_dataset.csv')
df.head()
df.shape

,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,35900,37.0,Male,Other,Strategy,23.929404,NaN,Hard,3,124,99,18,Medium
1,27085,25.0,Male,NaN,Action,22.755168,1.0,Easy,14,84,84,12,Medium
2,39595,24.0,Female,Europe,Simulation,19.505292,0.0,Hard,3,172,9,18,Medium
3,37440,26.0,Female,Europe,RPG,11.009645,NaN,NaN,3,83,36,43,Low
4,22882,17.0,Female,USA,RPG,0.581039,1.0,Medium,5,163,9,24,Medium


(10000, 13)

In [331]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PlayerID                   10000 non-null  int64  
 1   Age                        9200 non-null   float64
 2   Gender                     10000 non-null  str    
 3   Location                   9202 non-null   str    
 4   GameGenre                  10000 non-null  str    
 5   PlayTimeHours              10000 non-null  float64
 6   InGamePurchases            9107 non-null   float64
 7   GameDifficulty             9154 non-null   str    
 8   SessionsPerWeek            10000 non-null  int64  
 9   AvgSessionDurationMinutes  10000 non-null  int64  
 10  PlayerLevel                10000 non-null  int64  
 11  AchievementsUnlocked       10000 non-null  int64  
 12  EngagementLevel            10000 non-null  str    
dtypes: float64(3), int64(5), str(5)
memory usage: 1015.8 KB


In [332]:
df.select_dtypes(['object', 'str']).columns

Index(['Gender', 'Location', 'GameGenre', 'GameDifficulty', 'EngagementLevel'], dtype='str')

In [333]:
df[(df['Gender'] == 'Male')
   & (df['Location'] == 'Europe')
   & (df['InGamePurchases'] == 1)].shape

(299, 13)

In [334]:
df[(df['Age'] < 18)
   & (df['PlayTimeHours'] > 10)].shape

(453, 13)

In [335]:
df.isnull().sum().sum()

np.int64(3337)

## Split into feature matrix and target vector

In [336]:
X = df.drop(columns=['EngagementLevel'])
y = df['EngagementLevel']

X.shape
y.shape

(10000, 12)

(10000,)

## Train test splitting

In [337]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(8000, 12)
(8000,)
(2000, 12)
(2000,)


In [338]:
y_train.value_counts()

EngagementLevel
Medium    3983
Low       2021
High      1996
Name: count, dtype: int64

## Imputation

In [339]:
X_train.isnull().sum().sum()
X_test.isnull().sum().sum()

np.int64(2670)

np.int64(667)

In [340]:
strategies = {
    'Age': df['Age'].mean(),
    'Location': 'Other',
    'GameDifficulty': df['GameDifficulty'].mode()[0],
    'InGamePurchases': 0
}

X_train = X_train.fillna(value=strategies)
X_test = X_test.fillna(value=strategies)

X_train.isnull().sum().sum()
X_test.isnull().sum().sum()

np.int64(0)

np.int64(0)

In [341]:
X_test['Age'].sum()

np.float64(63587.973913043475)

## Preprocessing

In [342]:
X_train = X_train.drop(columns=['PlayerID'])
X_test = X_test.drop(columns=['PlayerID'])

In [343]:
from sklearn.preprocessing import OrdinalEncoder

difficulty_order = ['Easy', 'Medium', 'Hard']
oe = OrdinalEncoder(categories=[difficulty_order])
X_train[['GameDifficulty']] = oe.fit_transform(X_train[['GameDifficulty']])
X_test[['GameDifficulty']] = oe.transform(X_test[['GameDifficulty']])

In [344]:
from sklearn.preprocessing import OneHotEncoder

nominal_cols = ['Gender', 'Location', 'GameGenre']
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_train = ohe.fit_transform(X_train[nominal_cols])
ohe_test = ohe.transform(X_test[nominal_cols])

ohe_train.shape
ohe_test.shape

(8000, 8)

(2000, 8)

In [345]:
ohe_cols = ohe.get_feature_names_out(['Gender', 'Location', 'GameGenre'])
ohe_cols

array(['Gender_Male', 'Location_Europe', 'Location_Other', 'Location_USA',
       'GameGenre_RPG', 'GameGenre_Simulation', 'GameGenre_Sports',
       'GameGenre_Strategy'], dtype=object)

In [346]:
for col in nominal_cols:
  print(col, len(X_train[col].value_counts())-1)

Gender 1
Location 3
GameGenre 4


In [347]:
ohe_train_df = pd.DataFrame(ohe_train, columns=ohe_cols)
ohe_test_df = pd.DataFrame(ohe_test, columns=ohe_cols)

ohe_train_df.head(1)
ohe_test_df.head(1)

,Gender_Male,Location_Europe,Location_Other,Location_USA,GameGenre_RPG,GameGenre_Simulation,GameGenre_Sports,GameGenre_Strategy
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,Gender_Male,Location_Europe,Location_Other,Location_USA,GameGenre_RPG,GameGenre_Simulation,GameGenre_Sports,GameGenre_Strategy
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [348]:
X_train_enc = pd.concat([
    X_train.drop(columns=nominal_cols).reset_index(drop=True),
    ohe_train_df.reset_index(drop=True)
], axis=1)

X_test_enc = pd.concat([
    X_test.drop(columns=nominal_cols).reset_index(drop=True),
    ohe_test_df.reset_index(drop=True)
], axis=1)

X_train_enc.head(1)
X_train_enc.shape

X_test_enc.head(1)
X_test_enc.shape

,Age,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,Gender_Male,Location_Europe,Location_Other,Location_USA,GameGenre_RPG,GameGenre_Simulation,GameGenre_Sports,GameGenre_Strategy
0,34.0,15.777862,0.0,1.0,5,76,80,33,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


(8000, 16)

,Age,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,Gender_Male,Location_Europe,Location_Other,Location_USA,GameGenre_RPG,GameGenre_Simulation,GameGenre_Sports,GameGenre_Strategy
0,38.0,13.585655,0.0,0.0,18,141,96,19,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


(2000, 16)

In [349]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)

type(X_train_scaled)
type(X_test_scaled)

X_train_scaled.shape
X_test_scaled.shape

numpy.ndarray

numpy.ndarray

(8000, 16)

(2000, 16)

In [350]:
X_test_scaled[:5].sum()

np.float64(-7.171767806299545)